# Task 3 — Analytics Charts
**BMIT3003 Data Warehouse Technology** · Domain A (XS): Sales & Product

Run every cell top to bottom. The final cell downloads all 15 charts as a zip.

---

### Three rules baked into these charts

1. **2026 is January–August only.** Where it appears it is hatched and labelled, and it is excluded from every year-on-year calculation.
2. **Charts 14 and 15 show a discontinuity, not a trend.** 2024 is where the reconstructed history meets the original operational records.
3. **Colour is a CVD-validated categorical palette** assigned in fixed slot order, never more than three series on one plot, and no chart uses a dual axis.


## 1 · Setup

Imports, the CVD-validated palette, and a matplotlib style tuned for print. Charts save at 220 DPI so they stay sharp when placed in a Word document.


In [ ]:
# %% ===========================  CELL 1 - SETUP  =============================
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
import os

os.makedirs("charts", exist_ok=True)

# --- Validated categorical palette (fixed slot order, never cycled) ----------
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"      # blue, orange, aqua
RED, BLUE  = "#e34948", "#2a78d6"                  # diverging pair
INK        = "#1a1a19"
INK_SOFT   = "#52514e"
INK_MUTE   = "#8a8985"
GRID       = "#e6e5e1"
SURFACE    = "#ffffff"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,                 # print-quality for a Word report
    "savefig.bbox": "tight",
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "font.family": ["DejaVu Sans"],
    "font.size": 10,
    "axes.edgecolor": GRID,
    "axes.linewidth": 0.8,
    "axes.labelcolor": INK_SOFT,
    "axes.titlesize": 12.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": INK,
    "axes.titlepad": 14,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.frameon": False,
    "legend.fontsize": 9.5,
})

def clean(ax, xgrid=False, ygrid=True):
    """Recessive axes: drop the box, keep one grid direction."""
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.xaxis.grid(xgrid)
    ax.yaxis.grid(ygrid)
    ax.tick_params(length=0)

def subtitle(ax, text):
    """Call AFTER set_title. Re-pads the title so the two never collide."""
    ax.set_title(ax.get_title(), pad=30)
    ax.text(0, 1.012, text, transform=ax.transAxes, fontsize=9.5,
            color=INK_MUTE, ha="left", va="bottom")

rm = FuncFormatter(lambda v, p: f"{v:,.0f}")
pct = FuncFormatter(lambda v, p: f"{v:.0f}%")

def save(fig, name):
    fig.savefig(f"charts/{name}.png", facecolor=SURFACE)
    print("saved charts/" + name + ".png")

print("Setup complete.")

## 2 · Data

Every figure below is transcribed from the output of `Task 3/task3_xs_reports.sql`. Nothing is invented or simulated here — if you re-run the SQL after a reload, update these lists and re-run the notebook.

`PART_YEAR = 2026` and `BOUNDARY = 2024` are the two constants that keep the caveats honest.


In [ ]:
# %% ===========================  CELL 2 - DATA  ==============================
# ---- Exhibit 1.1 / 1.2 : yearly headline ------------------------------------
year        = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
orders      = [250, 330, 430, 540, 330, 450, 680, 900, 1100, 1350, 1002]
revenue     = [14337, 19155, 26101, 32144, 19640, 28213, 47682, 73699, 93023, 124891, 81865]
categories  = [6, 7, 8, 9, 10, 10, 11, 12, 12, 12, 12]
items_sold  = [28, 32, 36, 39, 43, 47, 49, 54, 54, 54, 54]
avg_basket  = [57.35, 58.05, 60.70, 59.53, 59.51, 62.70, 70.12, 81.89, 84.57, 92.51, 81.70]
yoy_pct     = [None, 33.6, 36.3, 23.2, -38.9, 43.7, 69.0, 54.6, 26.2, 34.3, None]

PART_YEAR = 2026   # January-August only

# ---- Exhibit 1.3 : revenue by category by year ------------------------------
cat_rows = [
 (2016,"Bakery",1352),(2016,"Beverages",1772),(2016,"Dairy and Eggs",2960),
 (2016,"Frozen Food",3690),(2016,"Rice and Noodles",2487),(2016,"Snacks",2076),
 (2017,"Bakery",1570),(2017,"Beverages",2396),(2017,"Cooking Essentials",3087),
 (2017,"Dairy and Eggs",3942),(2017,"Frozen Food",3602),(2017,"Rice and Noodles",2317),
 (2017,"Snacks",2242),
 (2018,"Bakery",1880),(2018,"Beverages",2704),(2018,"Canned Food",1811),
 (2018,"Cooking Essentials",4137),(2018,"Dairy and Eggs",4165),(2018,"Frozen Food",5630),
 (2018,"Rice and Noodles",3153),(2018,"Snacks",2621),
 (2019,"Bakery",2171),(2019,"Beverages",3186),(2019,"Canned Food",3132),
 (2019,"Cooking Essentials",3846),(2019,"Dairy and Eggs",4876),(2019,"Frozen Food",6213),
 (2019,"Personal Care",2023),(2019,"Rice and Noodles",3875),(2019,"Snacks",2821),
 (2020,"Bakery",1290),(2020,"Beverages",1540),(2020,"Canned Food",1729),
 (2020,"Cooking Essentials",2473),(2020,"Dairy and Eggs",2602),(2020,"Frozen Food",3074),
 (2020,"Household Cleaning",351),(2020,"Personal Care",3039),
 (2020,"Rice and Noodles",1956),(2020,"Snacks",1586),
 (2021,"Bakery",1333),(2021,"Beverages",1999),(2021,"Canned Food",2480),
 (2021,"Cooking Essentials",3553),(2021,"Dairy and Eggs",3533),(2021,"Frozen Food",4429),
 (2021,"Household Cleaning",3132),(2021,"Personal Care",3113),
 (2021,"Rice and Noodles",2595),(2021,"Snacks",2045),
 (2022,"Baby Products",9695),(2022,"Bakery",2260),(2022,"Beverages",2604),
 (2022,"Canned Food",3196),(2022,"Cooking Essentials",4868),(2022,"Dairy and Eggs",3704),
 (2022,"Frozen Food",5671),(2022,"Household Cleaning",4518),(2022,"Personal Care",5064),
 (2022,"Rice and Noodles",3517),(2022,"Snacks",2586),
 (2023,"Baby Products",15237),(2023,"Bakery",2598),(2023,"Beverages",3598),
 (2023,"Canned Food",4997),(2023,"Cooking Essentials",5807),(2023,"Dairy and Eggs",5505),
 (2023,"Frozen Food",6662),(2023,"Household Cleaning",4512),(2023,"Personal Care",7385),
 (2023,"Pet Care",10566),(2023,"Rice and Noodles",4165),(2023,"Snacks",2667),
 (2024,"Baby Products",18182),(2024,"Bakery",3611),(2024,"Beverages",5599),
 (2024,"Canned Food",5784),(2024,"Cooking Essentials",7751),(2024,"Dairy and Eggs",8091),
 (2024,"Frozen Food",9495),(2024,"Household Cleaning",6982),(2024,"Personal Care",7408),
 (2024,"Pet Care",11440),(2024,"Rice and Noodles",5134),(2024,"Snacks",3547),
 (2025,"Baby Products",27619),(2025,"Bakery",4546),(2025,"Beverages",5860),
 (2025,"Canned Food",7066),(2025,"Cooking Essentials",8454),(2025,"Dairy and Eggs",9392),
 (2025,"Frozen Food",11385),(2025,"Household Cleaning",8903),(2025,"Personal Care",9667),
 (2025,"Pet Care",18969),(2025,"Rice and Noodles",8724),(2025,"Snacks",4305),
 (2026,"Baby Products",16073),(2026,"Bakery",3099),(2026,"Beverages",4033),
 (2026,"Canned Food",4065),(2026,"Cooking Essentials",6538),(2026,"Dairy and Eggs",7466),
 (2026,"Frozen Food",8135),(2026,"Household Cleaning",5814),(2026,"Personal Care",6580),
 (2026,"Pet Care",11917),(2026,"Rice and Noodles",4697),(2026,"Snacks",3448),
]
cat = pd.DataFrame(cat_rows, columns=["year", "category", "revenue"])

# ---- Exhibit 1.4 : category entry ------------------------------------------
entry = pd.DataFrame([
 ("Frozen Food",2016,5,67987,12.1),("Dairy and Eggs",2016,4,56235,10.0),
 ("Rice and Noodles",2016,4,42621,7.6),("Beverages",2016,5,35290,6.3),
 ("Snacks",2016,5,29943,5.3),("Bakery",2016,5,25708,4.6),
 ("Cooking Essentials",2017,5,50514,9.0),("Canned Food",2018,5,34259,6.1),
 ("Personal Care",2019,4,44281,7.9),("Household Cleaning",2020,5,34212,6.1),
 ("Baby Products",2022,4,86807,15.5),("Pet Care",2023,4,52892,9.4),
], columns=["category","first_year","items","lifetime_revenue","pct_of_total"])

# Cohort = the range-expansion story, and it keeps us to three series
def cohort(y):
    if y == 2016: return "Founding six (2016)"
    if y <= 2020: return "Added 2017-2020"
    return "Late entrants (2022-23)"
entry["cohort"] = entry["first_year"].apply(cohort)
COHORTS = ["Founding six (2016)", "Added 2017-2020", "Late entrants (2022-23)"]
COHORT_C = {COHORTS[0]: C1, COHORTS[1]: C2, COHORTS[2]: C3}
cat = cat.merge(entry[["category","cohort"]], on="category", how="left")

# ---- Exhibit 1.5 / 1.6 : seasonality ---------------------------------------
quarters   = ["Q1", "Q2", "Q3", "Q4"]
q_orders   = [1709, 1544, 1455, 1652]
q_index    = [107, 97, 92, 104]

q1_share = pd.DataFrame([
 ("Pet Care",29.5),("Dairy and Eggs",27.6),("Canned Food",27.4),("Snacks",26.1),
 ("Frozen Food",26.1),("Household Cleaning",25.7),("Bakery",25.7),
 ("Rice and Noodles",25.1),("Cooking Essentials",25.0),("Personal Care",24.6),
 ("Beverages",24.5),("Baby Products",20.9),
], columns=["category","q1_pct"])
STORE_Q1 = 1709 / (1709+1544+1455+1652) * 100      # store-wide Q1 share

# ---- Exhibit 2.1 / 2.3 : suppliers -----------------------------------------
sup = pd.DataFrame([
 ("LittleStar Baby Products",86807,15.5,"Baby Products",3094,82,89.9,-0.83),
 ("Polar Frozen Foods",      67987,12.1,"Frozen Food",  6831,219,198.4, 1.46),
 ("Fresh Dairy Farm",        56235,10.0,"Dairy and Eggs",5457,163,158.5, 0.36),
 ("PetJoy Trading",          52892, 9.4,"Pet Care",     2836, 99, 82.4, 1.83),
 ("Selera Cooking Products", 50514, 9.0,"Cooking Ess.", 6182,166,179.6,-1.01),
 ("CarePlus Consumer Goods", 44281, 7.9,"Personal Care",3891,103,113.0,-0.94),
 ("Padi Emas Rice Mills",    42621, 7.6,"Rice & Noodles",5356,152,155.6,-0.29),
 ("Sunshine Beverages",      35290, 6.3,"Beverages",    6679,165,194.0,-2.08),
 ("Ocean Canned Food",       34259, 6.1,"Canned Food",  5829,177,169.3, 0.59),
 ("KleenHome Supplies",      34212, 6.1,"Household Cl.",4536,172,131.7, 3.51),
 ("Snacko Food Industries",  29943, 5.3,"Snacks",       5869,148,170.5,-1.72),
 ("Gardenview Bakery",       25708, 4.6,"Bakery",       6652,190,193.2,-0.23),
], columns=["supplier","revenue","pct","category","units_sold",
            "returned","expected","z"])
sup["cum_pct"] = sup["pct"].cumsum()
# A return line takes back 1-3 units, so returns arrive in clusters. The plain
# Poisson SD assumes single units and therefore understates true variance by
# about 1.5x. Dividing by that factor is the honest z.
CLUSTER_ADJ = 1.53
sup["z_adj"] = sup["z"] / CLUSTER_ADJ

# ---- Exhibit 2.4 : return reasons ------------------------------------------
reasons = pd.DataFrame([
 ("Wrong Item","Fulfilment",243,25.7),("Broken","Product Quality",235,24.9),
 ("Expired","Product Quality",233,24.7),("Missing","Fulfilment",233,24.7),
], columns=["reason","group","lines","pct"])

# ---- Exhibit 3.1 / 3.2 / 3.3 / 3.4 : channel -------------------------------
online_pct = [17.6, 20.6, 27.9, 30.4, 34.5, 37.1, 42.6, 42.7, 42.8, 44.9, 50.5]

region = pd.DataFrame([
 ("Northern",876,48.5,89.46),("East Malaysia",287,46.0,83.86),
 ("Central",1403,45.9,87.47),("East Coast",299,44.8,85.37),
 ("Southern",587,42.2,83.62),
], columns=["region","orders","online_pct","avg_basket"])

tier = pd.DataFrame([
 ("VIP",2218,44.9,76.63),("Normal",2776,43.1,77.27),("Non-Member",2368,31.4,74.45),
], columns=["tier","orders","online_pct","avg_basket"])

basket = pd.DataFrame([
 (2022,70.66,69.72),(2023,76.55,85.86),(2024,84.35,84.73),
 (2025,92.02,92.91),(2026,81.54,81.87),
], columns=["year","online","walkin"])

# ---- Confounded series (charted to SHOW the break, not a trend) -------------
return_pct = [2.49, 2.80, 1.77, 2.45, 2.05, 2.57, 2.60, 2.83, 3.21, 3.86, 2.70]
lead_days  = [3.70, 3.97, 4.18, 3.74, 4.46, 4.39, 4.02, 3.99, 2.60, 2.87, 3.08]
cancel_pct = [6.90, 8.82, 4.69, 12.37, 14.29, 11.58, 12.24, 9.13, 5.73, 5.15, 8.83]
BOUNDARY   = 2024

print("Data loaded:", len(cat), "category-year rows,", len(sup), "suppliers.")

## 3 · Report 1 — Growth and the 2020 shock

**Chart 01** revenue by year, with 2020 de-emphasised and 2026 hatched as a part year.

**Chart 02** year-on-year change as a diverging bar — the right form for polarity data. 2026 is excluded because a part year cannot carry a YoY percentage.


In [ ]:
# %% ==================  CELL 3 - R1: GROWTH AND THE SHOCK  ===================
fig, ax = plt.subplots(figsize=(9, 4.6))
bars = ax.bar(year, revenue, color=C1, width=0.66, zorder=3)
bars[year.index(2020)].set_color(INK_MUTE)          # the shock year, de-emphasised
bars[year.index(PART_YEAR)].set_color(C1)
bars[year.index(PART_YEAR)].set_hatch("////")
bars[year.index(PART_YEAR)].set_alpha(0.45)

ax.set_title("Revenue grew 8.7x while the range doubled")
subtitle(ax, "Net sales revenue, RM. 2020 highlighted; 2026 is January-August only.")
ax.yaxis.set_major_formatter(rm); ax.set_ylabel("Net revenue (RM)")
ax.set_xticks(year); clean(ax)
ax.annotate("2020: -38.9%", xy=(2020, 19640), xytext=(2020, 46000),
            ha="center", fontsize=9.5, color=INK_SOFT,
            arrowprops=dict(arrowstyle="-", color=INK_MUTE, lw=1))
ax.annotate("part year", xy=(2026, 81865), xytext=(2026, 104000),
            ha="center", fontsize=9, color=INK_MUTE,
            arrowprops=dict(arrowstyle="-", color=INK_MUTE, lw=1))
save(fig, "01_revenue_by_year"); plt.show()

# --- Year-on-year: a diverging bar, which is what polarity data wants --------
yy = [(y, v) for y, v in zip(year, yoy_pct) if v is not None]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar([y for y, _ in yy], [v for _, v in yy], width=0.66, zorder=3,
       color=[BLUE if v > 0 else RED for _, v in yy])
ax.axhline(0, color=INK_SOFT, lw=1)
ax.set_title("A 38.9% collapse in 2020, then two years to recover")
subtitle(ax, "Year-on-year change in net revenue. 2026 excluded: a part year carries no YoY.")
ax.yaxis.set_major_formatter(pct); ax.set_ylabel("YoY change")
ax.set_xticks([y for y, _ in yy]); clean(ax)
for y, v in yy:
    ax.text(y, v + (3 if v > 0 else -6), f"{v:+.1f}%", ha="center",
            fontsize=8.5, color=INK_SOFT)
save(fig, "02_yoy_growth"); plt.show()

## 4 · Report 1 — Where the growth came from

**Chart 03** collapses twelve categories into three entry cohorts. The green wedge appearing in 2022 is the report's thesis in one shape.

**Chart 04** entry year against lifetime contribution. Baby Products entered last and leads.


In [ ]:
# %% ==============  CELL 4 - R1: WHERE THE GROWTH CAME FROM  =================
piv = (cat.pivot_table(index="year", columns="cohort", values="revenue",
                       aggfunc="sum").fillna(0)[COHORTS])
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.stackplot(piv.index, [piv[c] for c in COHORTS],
             colors=[COHORT_C[c] for c in COHORTS], labels=COHORTS,
             edgecolor=SURFACE, linewidth=2)          # 2px surface gap between fills
ax.set_title("Two late categories drive a quarter of all revenue")
subtitle(ax, "Net revenue by the year each category entered the range. RM.")
ax.yaxis.set_major_formatter(rm); ax.set_ylabel("Net revenue (RM)")
ax.set_xticks(year); ax.set_xlim(2016, 2026); clean(ax)
ax.axvline(PART_YEAR, color=INK_MUTE, lw=1, ls=":", zorder=5)
ax.text(2025.95, ax.get_ylim()[1]*0.97, "2026 part year ", ha="right", va="top",
        fontsize=8.5, color=INK_MUTE)
ax.legend(loc="upper left", ncol=1)
save(fig, "03_cohort_stack"); plt.show()

# --- Entry year vs lifetime contribution -------------------------------------
e = entry.sort_values(["first_year", "lifetime_revenue"], ascending=[True, True])
fig, ax = plt.subplots(figsize=(9, 5.2))
ypos = np.arange(len(e))
ax.hlines(ypos, 0, e["lifetime_revenue"], color=GRID, lw=1.5, zorder=2)
ax.scatter(e["lifetime_revenue"], ypos, s=110, zorder=3,
           color=[COHORT_C[c] for c in e["cohort"]])
ax.set_yticks(ypos); ax.set_yticklabels(e["category"])
ax.set_title("Baby Products entered in 2022 and is already the largest category")
subtitle(ax, "Lifetime net revenue by category, RM. Colour marks the entry cohort.")
ax.xaxis.set_major_formatter(rm); ax.set_xlabel("Lifetime net revenue (RM)")
clean(ax, xgrid=True, ygrid=False)
for i, r in enumerate(e.itertuples()):
    ax.text(r.lifetime_revenue + 1800, i, f"{r.first_year}  ·  {r.pct_of_total}%",
            va="center", fontsize=8.5, color=INK_SOFT)
ax.set_xlim(0, 104000)
handles = [plt.Line2D([], [], marker="o", ls="", ms=9, color=COHORT_C[c], label=c)
           for c in COHORTS]
ax.legend(handles=handles, loc="lower right")
save(fig, "04_category_entry"); plt.show()

## 5 · Report 1 — Seasonality

**Chart 05** quarterly index, 100 = the average quarter.

**Chart 06** the Q1 share of each category against the store-wide line. The tight clustering is the finding: seasonality lifts traffic, not any particular basket.


In [ ]:
# %% ====================  CELL 5 - R1: SEASONALITY  ==========================
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.bar(quarters, q_index, color=C1, width=0.6, zorder=3)
ax.axhline(100, color=INK_SOFT, lw=1.2, ls="--", zorder=4)
ax.text(3.42, 100.4, "average quarter", ha="right", fontsize=8.5, color=INK_SOFT)
ax.set_title("Q1 and Q4 carry the year")
subtitle(ax, "Orders indexed to the average quarter = 100. 2026 excluded.")
ax.set_ylim(85, 112); ax.set_ylabel("Order index"); clean(ax)
for q, v in zip(quarters, q_index):
    ax.text(q, v + 0.8, str(v), ha="center", fontsize=10, color=INK, weight="bold")
save(fig, "05_quarterly_index"); plt.show()

# --- Is seasonality category-specific? ---------------------------------------
qs = q1_share.sort_values("q1_pct")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(qs["category"], qs["q1_pct"], color=C1, height=0.62, zorder=3)
ax.axvline(STORE_Q1, color=INK_SOFT, lw=1.2, ls="--", zorder=4)
ax.text(STORE_Q1 + 0.25, -0.6, f"store-wide {STORE_Q1:.1f}%",
        fontsize=8.5, color=INK_SOFT)
ax.set_title("Seasonality is store-wide, not category-specific")
subtitle(ax, "Share of each category's revenue falling in Q1. 2026 excluded.")
ax.xaxis.set_major_formatter(pct); ax.set_xlim(0, 33)
ax.set_xlabel("Q1 share of category revenue"); clean(ax, xgrid=True, ygrid=False)
for i, r in enumerate(qs.itertuples()):
    ax.text(r.q1_pct + 0.4, i, f"{r.q1_pct}%", va="center",
            fontsize=8.5, color=INK_SOFT)
save(fig, "06_q1_share_by_category"); plt.show()

## 6 · Report 2 — Concentration and return exposure

**Chart 07** supplier revenue, each bar labelled with the category that supplier *solely* supplies.

**Chart 08** the key exhibit. Orange is the raw Poisson z-score, blue is adjusted for the fact that returns arrive in clusters of 1–3 units. Bands are ±2 and ±2.87 (Bonferroni, 12 simultaneous tests).

**Chart 09** return reasons against a 25% even-split line.


In [ ]:
# %% ================  CELL 6 - R2: CONCENTRATION AND RISK  ===================
s = sup.sort_values("revenue")
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.barh(s["supplier"], s["revenue"], color=C1, height=0.62, zorder=3)
ax.set_title("Every supplier is the sole source of exactly one category")
subtitle(ax, "Lifetime net revenue, RM. The category each supplier solely supplies is named.")
ax.xaxis.set_major_formatter(rm); ax.set_xlabel("Lifetime net revenue (RM)")
clean(ax, xgrid=True, ygrid=False); ax.set_xlim(0, 112000)
for i, r in enumerate(s.itertuples()):
    ax.text(r.revenue + 1600, i, f"{r.pct}%  ·  {r.category}",
            va="center", fontsize=8.5, color=INK_SOFT)
save(fig, "07_supplier_pareto"); plt.show()

# --- THE key exhibit: is any supplier a real quality outlier? ----------------
sz = sup.sort_values("z")
ypos = np.arange(len(sz))
fig, ax = plt.subplots(figsize=(9, 5.4))
ax.axvspan(-2.87, 2.87, color=C1, alpha=0.07, zorder=1)
ax.axvspan(-2, 2, color=C1, alpha=0.10, zorder=1)
ax.axvline(0, color=INK_SOFT, lw=1.2, zorder=2)
for x in (-2, 2):
    ax.axvline(x, color=INK_MUTE, lw=1, ls="--", zorder=2)
ax.hlines(ypos, sz["z_adj"], sz["z"], color=GRID, lw=1.6, zorder=3)
ax.scatter(sz["z"], ypos, s=95, color=C2, zorder=5, label="Raw z (Poisson)")
ax.scatter(sz["z_adj"], ypos, s=95, color=C1, zorder=5,
           label="Adjusted for return clustering")
ax.set_yticks(ypos); ax.set_yticklabels(sz["supplier"])
ax.set_title("No supplier's return rate departs far enough to act on")
subtitle(ax, "Deviation from expected returns, in standard deviations. "
             "Shaded bands: +/-2 and +/-2.87 (Bonferroni, 12 tests).")
ax.set_xlabel("z-score"); ax.set_xlim(-4.6, 4.6)
clean(ax, xgrid=True, ygrid=False)
ax.legend(loc="lower right")
ax.text(-4.45, len(sz) - 1.15,
        "KleenHome is the only supplier outside +/-2 on the raw score.\n"
        "Adjusted for clustering it falls to 2.29 - inside the band that\n"
        "12 simultaneous comparisons require.",
        fontsize=8.5, color=INK_SOFT, va="top", linespacing=1.5)
save(fig, "08_supplier_zscore"); plt.show()

# --- Return reasons: the point is how FLAT this is ---------------------------
r = reasons.sort_values("pct")
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.barh(r["reason"], r["pct"],
        color=[C2 if g == "Fulfilment" else C1 for g in r["group"]],
        height=0.6, zorder=3)
ax.axvline(25, color=INK_SOFT, lw=1.2, ls="--", zorder=4)
ax.text(25.15, -0.62, "25% = perfectly even", fontsize=8.5, color=INK_SOFT)
ax.set_title("No dominant failure mode")
subtitle(ax, "Share of return lines by reason. Orange = Fulfilment, blue = Product Quality.")
ax.xaxis.set_major_formatter(pct); ax.set_xlim(0, 30)
clean(ax, xgrid=True, ygrid=False)
for i, row in enumerate(r.itertuples()):
    ax.text(row.pct + 0.3, i, f"{row.pct}%", va="center", fontsize=8.5, color=INK_SOFT)
save(fig, "09_return_reasons"); plt.show()

## 7 · Report 3 — Channel migration

**Chart 10** online share crossing half of all orders.

**Chart 11** membership split into two panels — never a dual axis.

**Chart 12** online and walk-in baskets tracking within RM 1.

**Chart 13** regions, restricted to 2024–2026 so every branch was trading throughout.


In [ ]:
# %% =================  CELL 7 - R3: CHANNEL MIGRATION  =======================
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(year, online_pct, color=C1, lw=2.4, marker="o", ms=7,
        mfc=SURFACE, mew=2.2, zorder=4)
ax.axhline(50, color=INK_MUTE, lw=1, ls="--", zorder=2)
ax.text(2016, 50.7, "half of all orders", fontsize=8.5, color=INK_SOFT)
ax.set_title("Online crossed half of all orders in 2026")
subtitle(ax, "Online share of orders. Share is unaffected by 2026 being a part year.")
ax.yaxis.set_major_formatter(pct); ax.set_ylabel("Online share of orders")
ax.set_xticks(year); ax.set_ylim(10, 58); clean(ax)
for y, v in [(2016, 17.6), (2026, 50.5)]:
    ax.annotate(f"{v}%", xy=(y, v), xytext=(0, 13), textcoords="offset points",
                ha="center", fontsize=10, weight="bold", color=INK)
save(fig, "10_online_share"); plt.show()

# --- Membership: two measures, so two panels. Never a dual axis. -------------
t = tier.sort_values("online_pct")
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.9))
axes[0].barh(t["tier"], t["online_pct"],
             color=[C2 if x == "Non-Member" else C1 for x in t["tier"]],
             height=0.6, zorder=3)
axes[0].set_title("Members buy online", fontsize=11.5)
axes[0].xaxis.set_major_formatter(pct); axes[0].set_xlim(0, 55)
clean(axes[0], xgrid=True, ygrid=False)
for i, row in enumerate(t.itertuples()):
    axes[0].text(row.online_pct + 1, i, f"{row.online_pct}%", va="center",
                 fontsize=9, color=INK_SOFT)

axes[1].barh(t["tier"], t["avg_basket"],
             color=[C2 if x == "Non-Member" else C1 for x in t["tier"]],
             height=0.6, zorder=3)
axes[1].set_title("...but do not spend more", fontsize=11.5)
axes[1].set_xlim(0, 95); clean(axes[1], xgrid=True, ygrid=False)
for i, row in enumerate(t.itertuples()):
    axes[1].text(row.avg_basket + 1.6, i, f"RM {row.avg_basket:.2f}", va="center",
                 fontsize=9, color=INK_SOFT)
fig.suptitle("Membership drives channel, not spend", fontsize=12.5,
             fontweight="bold", color=INK, y=1.06, x=0.02, ha="left")
save(fig, "11_membership_tier"); plt.show()

# --- Online vs walk-in basket: the lines sit on top of each other ------------
fig, ax = plt.subplots(figsize=(8.4, 4.2))
ax.plot(basket["year"], basket["online"], color=C1, lw=2.4, marker="o", ms=7,
        mfc=SURFACE, mew=2.2, label="Online", zorder=4)
ax.plot(basket["year"], basket["walkin"], color=C2, lw=2.4, marker="s", ms=7,
        mfc=SURFACE, mew=2.2, label="Walk-in", zorder=4)
ax.set_title("The migration substitutes, it does not grow the basket")
subtitle(ax, "Average basket value by channel, RM. The two lines track within RM 1.")
ax.set_ylabel("Average basket (RM)"); ax.set_xticks(basket["year"])
clean(ax); ax.legend(loc="lower right")
save(fig, "12_basket_by_channel"); plt.show()

# --- Region, like-for-like ---------------------------------------------------
rg = region.sort_values("online_pct")
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(rg["region"], rg["online_pct"], color=C1, height=0.6, zorder=3)
ax.set_title("Regional differences are too small to act on")
subtitle(ax, "Online share, 2024-2026 only - all twelve branches trading throughout.")
ax.xaxis.set_major_formatter(pct); ax.set_xlim(0, 58)
clean(ax, xgrid=True, ygrid=False)
for i, row in enumerate(rg.itertuples()):
    ax.text(row.online_pct + 0.8, i, f"{row.online_pct}%   (n={row.orders:,})",
            va="center", fontsize=8.5, color=INK_SOFT)
save(fig, "13_region_channel"); plt.show()

## 8 · The confounds

These two charts exist to show a **discontinuity, not a trend**. The series is split at 2024 and coloured by data source.

Do not describe either as an improvement or a deterioration — the step sits exactly where the reconstructed history meets the original operational records.


In [ ]:
# %% ==============  CELL 8 - THE CONFOUNDS (show the break)  =================
# These two charts exist to demonstrate a DISCONTINUITY. Do not describe
# either as a trend.
for name, series, ylab, title, fname in [
    ("Return rate", return_pct, "Units returned as % of units sold",
     "Return rate does not rise - the datasets change", "14_confound_return_rate"),
    ("Delivery lead time", lead_days, "Average lead time (days)",
     "Fulfilment does not improve - the datasets change", "15_confound_lead_time"),
]:
    fig, ax = plt.subplots(figsize=(9, 4.2))
    pre = [i for i, y in enumerate(year) if y < BOUNDARY]
    post = [i for i, y in enumerate(year) if y >= BOUNDARY]
    ax.plot([year[i] for i in pre], [series[i] for i in pre], color=C1, lw=2.4,
            marker="o", ms=7, mfc=SURFACE, mew=2.2, label="Reconstructed history", zorder=4)
    ax.plot([year[i] for i in post], [series[i] for i in post], color=C2, lw=2.4,
            marker="o", ms=7, mfc=SURFACE, mew=2.2, label="Original records (blended)", zorder=4)
    ax.axvline(BOUNDARY - 0.5, color=INK_SOFT, lw=1.4, ls="--", zorder=3)
    ax.text(BOUNDARY - 0.42, ax.get_ylim()[1], " dataset boundary", va="top",
            fontsize=9, color=INK_SOFT)
    ax.set_title(title)
    subtitle(ax, "The step sits exactly where the two data sources meet, not across the series.")
    ax.set_ylabel(ylab); ax.set_xticks(year); clean(ax)
    ax.legend(loc="upper left")
    save(fig, fname); plt.show()

## 9 · Download

Zips every PNG and downloads it. Outside Colab the files are left in `./charts/`.


In [ ]:
# %% ===================  CELL 9 - DOWNLOAD EVERYTHING  =======================
import shutil
shutil.make_archive("task3_charts", "zip", "charts")
print("\nCharts written:")
for f in sorted(os.listdir("charts")):
    print("  ", f)

try:
    from google.colab import files
    files.download("task3_charts.zip")
except Exception as e:
    print("\nNot running in Colab - the PNGs are in ./charts/  (", e, ")")